# Class 2 — Modules, Packages, Random, and File Handling

**Duration:** 2 hours

## Learning objectives

By the end of this class, you should be able to:

- Write reusable functions with parameters and return values.
- Organize Python code into multiple modules.
- Import functions from other Python files.
- Explain packages, `pip`, and virtual environments.
- Generate simulated sensor readings using `random`.
- Read and write text and CSV files.
- Handle errors using `try` and `except`.
- Use Git and GitHub to manage your project.

We will use sensor examples related to ESP32 embedded programming.

**Note:** This lesson uses functions and modules, not classes or object-oriented programming.

## 1. Python recap — loops, lists, and functions

A list can store multiple sensor readings. A function can process those readings, and a loop can repeat an operation for every value.

Run the next cell and observe the output.

In [ ]:
def report_angles(angles):
    for angle in angles:
        print(angle)


angles = [-180, -90, 0, 90, 180]

report_angles(angles)

## 2. Functions in more depth

Functions help us divide a problem into smaller, reusable steps.

### Parameters and default values

A default parameter value is used when an argument is not provided.

### `return` versus `print`

- `print()` displays information.
- `return` sends a result back to the code that called the function.

### Variable scope

Variables created inside a function are normally local to that function. It is usually clearer to pass information through parameters and return values than to rely on global variables.

You could add a threshold for the angle

In [ ]:
def report_high_angles(angles, threshold):
    for angle in angles:
        if angle > threshold:
            print(angle)


angles = [-180, -90, 0, 90, 135, 180]

report_high_angles(angles, 90)

In [ ]:
import math

def convert_to_radians(angle_degrees):
    return angle_degrees * math.pi / 180


print(convert_to_radians(90))
print(convert_to_radians(180))

In [ ]:
def average_angles(angles):
    if len(angles) == 0:
        raise ValueError("The list cannot be empty.")

    return sum(angles) / len(angles)


result = average_angles([])

print("Average angle:", result)

### Decomposing a problem

Instead of writing one long program, divide it into smaller functions.

For our sensor application, we can separate the tasks into:

1. Generating readings.
2. Calculating statistics.
3. Saving the readings.
4. Displaying the results.

The program's `main()` function can coordinate these steps.

We will implement these functions throughout this lesson.

## 3. Modules and imports

A **module** is typically a single Python file containing functions, variables, or classes.

Modules help us organize a project and reuse code.

For example, we might use two files:

- `sensor.py` contains reusable sensor functions.
- `main.py` imports and runs those functions.

The following cell creates `sensor.py` directly from this notebook.

In [ ]:
%%writefile sensor.py

import math

def convert_to_radians(angle_degrees):
    return angle_degrees * math.pi / 180


def is_above_threshold(value, threshold):
    return value > threshold


def classify_angle(value):
    if value < 0:
        return "low"
    elif value <= 90:
        return "normal"
    else:
        return "high"

We can now use the generated file to show you how to use the functionality described in that file

In [ ]:
import sensor
print(sensor.__file__)
print(dir(sensor))

In [ ]:

from sensor import (
    convert_to_radians,
    is_above_threshold,
    classify_angle,
)

angle = 75
radians = convert_to_radians(angle)

print(f"Angle: {angle:.2f} V")
print("Above threshold:", is_above_threshold(angle, 1.5))
print("Angle category:", classify_angle(25))

### Other ways to import modules

Python provides a standard library containing useful modules such as `random`, `csv`, and `math`.

Examples:

- `import random` lets us use functions with the `random.` prefix.
- `from math import sqrt` imports one particular function.

**Important:** Avoid naming your own files `random.py`, `csv.py`, or `math.py`. These names can interfere with imports from Python's standard library.

### Quick exercise

Add another function to `sensor.py` that converts Celsius to Fahrenheit:

\[
F = (C * 9/5) + 32
\]

Import the function and test it with a temperature of 25 °C.

## 4. Packages, pip, and virtual environments

Important terminology:

- **Module:** typically one Python file.
- **Package:** a way of organizing related Python modules.
- **Library:** a general term for reusable code.
- **pip:** a tool commonly used to install Python packages.
- **Virtual environment:** an isolated environment for a project's dependencies.

### Check your Python installation below


In [ ]:
import sys

print("Python version:")
print(sys.version)

print("Python executable:")
print(sys.executable)

## 5. Random numbers and sensor simulation

The `random` module is useful when testing a sensor application without physical hardware.

Three useful functions are:

- `random.uniform(a, b)`: generate a floating-point value in a range.
- `random.randint(a, b)`: generate an integer, including both endpoints.
- `random.seed(value)`: make pseudo-random results reproducible.

The simulated readings below are examples, not actual measurements from an ESP32.

In [ ]:
import random

angle = random.uniform(18.0, 30.0)
value = random.randint(0, 4095)

print("Angle:", round(angle, 2))
print("Value:", value)

In [ ]:
import random


def generate_readings(count, minimum=-180.0, maximum=180.0):
    if count <= 0:
        raise ValueError("Count must be greater than zero.")

    if minimum >= maximum:
        raise ValueError("Minimum must be less than maximum.")

    readings = []

    for _ in range(count):
        value = round(random.uniform(minimum, maximum), 2)
        readings.append(value)

    return readings


angles = generate_readings(10)

print(angles)

In [ ]:
random.seed(7)
first_run = [random.randint(0, 180) for _ in range(5)]

random.seed(7)
second_run = [random.randint(0, 180) for _ in range(5)]

print("First run:", first_run)
print("Second run:", second_run)
print("Same results:", first_run == second_run)

## 6. Files, CSV, and exceptions

Files allow our program to store data for later use.

Common file modes:

- `"w"` writes a file and replaces existing contents.
- `"a"` appends to an existing file.
- `"r"` reads a file.

Using `with open(...)` ensures that the file is closed automatically.

### Write a text file

In [ ]:
with open("notes.txt", "w", encoding="utf-8") as file:
    file.write("Sensor test started\n")
    file.write("These are simulated readings\n")

print("Text file saved.")

### Save sensor readings to CSV

CSV means comma-separated values. It is a simple format for exchanging sensor data and opening it in spreadsheet software.

We will store a sample number and its corresponding temperature.

In [ ]:
import csv


def save_readings(filename, readings):
    with open(filename, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["sample", "angle_degrees"])

        for sample_number, angle in enumerate(readings, start=1):
            writer.writerow([sample_number, angle])


readings = [90, 80.5, 130.7, -70]

save_readings("readings.csv", readings)

print("CSV file saved.")

We can also read stored data from a file

In [ ]:
def load_readings(filename):
    readings = []

    with open(filename, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)

        for row in reader:
            readings.append(float(row["angle_degrees"]))

    return readings


loaded_readings = load_readings("readings.csv")

print("Loaded readings:", loaded_readings)

### Handle common errors

User input may not be a valid number, and a file may not exist or be accessible.

Python lets us handle these cases with `try` and `except`.

- `ValueError` can occur when converting invalid text to a number or validating a value.
- `OSError` can occur when a file cannot be opened or written.

We should handle expected errors rather than silently ignoring every possible problem.
We also would like the program to continue, rather than stop the user for entering corrected information.

In [ ]:
while True:
    try:
        user_input = input(
            "How many readings should be generated? (Enter q to quit): "
        ).strip()

        # Stop the loop when the user enters q, quit, or exit
        if user_input.lower() in ("q", "quit", "exit"):
            print("Program stopped.")
            break

        count = int(user_input)

        if count <= 0:
            raise ValueError("Please enter a positive number.")

        readings = generate_readings(count)
        save_readings("readings.csv", readings)

        print(f"Saved {len(readings)} readings.")

    except ValueError as error:
        print("Invalid input:", error)

    except OSError as error:
        print("Could not access the file:", error)

### Exercise: Ask the user for sensor settings

Extend the program so that the user can enter:

1. The number of readings.
2. The minimum angle value.
3. The maximum angle value.

Validate the inputs, generate the readings, and save them to a CSV file.

The program should display a helpful error message for all invalid input.

In [ ]:
# your code here..
#user can enter:

#1. The number of readings.
#2. The minimum angle value.
#3. The maximum angle value.



#Validate the inputs
    #The program should display a helpful error message for all invalid input.

#generate the readings

#save them to a CSV file


## 7. GitHub workflow

Continue using the repository created in Class 1.

A typical workflow is:

1. Pull the latest changes.
2. Edit or add files.
3. Check which files have changed.
4. Stage the intended changes.
5. Commit with a meaningful message.
6. Push the commit to GitHub.
7. Update the README.

Run Git commands in the VS Code terminal, from the repository folder.

In [ ]:
# This cell is for reference only.
# Run these commands in the VS Code terminal, not as Python.

# git status
# git pull
# git add main.py sensor.py README.md requirements.txt
# git commit -m "Add sensor simulation and CSV handling"
# git push

### GitHub exercise

- Add `sensor.py` and `main.py` to your repository.
- Add `requirements.txt` and a README.
- Run the program and verify that `readings.csv` is created.
- Commit and push your changes.
- Open GitHub and confirm that the files are visible.

Your README should explain what the program does, how to install its dependencies, and how to run it.

## End-of-class mini task

Create a small sensor-data program using **functions and modules**.

The program must:

1. Ask the user how many angle readings to generate.
2. Ask the user for the minimum and maximum angle values.
3. Validate the input.
4. Generate simulated readings using `random`.
5. Save the data to CSV.
6. Load the saved data and display the count, minimum, and maximum.
7. Be committed and pushed to GitHub.

NumPy and Matplotlib are not required yet; both will be covered together in Class 3.

## End result

You can now structure a small Python program across multiple modules, write reusable functions, generate simulated sensor data, work with text and CSV files, handle common errors, and use GitHub to manage your progress.